In [1]:
import json
from html import escape
from pathlib import Path
import random

import numpy as np
import pandas as pd

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import load_tokenizer
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from transformers import PreTrainedTokenizer

# EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.1-qwen3-1.7b-likelihood-tokens"
# EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.2-qwen3-1.7b-likelihood-tokens-trunc"
EXP41_DIR = get_data_dir() / "experiments" / "exp4.1-likelihood-tokens" / "exp4.1.3-qwen3-1.7b-likelihood-tokens-trunc-v2"
EXP41_RESULTS_FILE = EXP41_DIR / "data" / "loglikelihood_pivotal_tokens.csv"

EXP22_DIR = get_data_dir() / "experiments" / "exp2.2-alt-tokens-on-pivotal-positions" / "exp2.2.1-qwen3-1.7b-alt-tokens-on-pivotal-positions"
EXP22_RESULTS_FILE = EXP22_DIR / "data" / "alternative_tokens.csv"

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'

# PREPROCESSED_FILE = Path("data/preprocessed.csv")
# PREPROCESSED_FILE = Path("data/preprocessed-trunc.csv")
PREPROCESSED_FILE = Path("data/preprocessed-trunc-v2.csv")

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [2]:
# tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

# prob_init_df = pd.read_csv(EXP22_RESULTS_FILE)
# prob_init_df = prob_init_df[["sample_id", "prob_init"]].rename(columns={"sample_id": "id"}).copy()
# traces_df = pd.read_csv(TRACES_FILE)
# traces_df = pd.merge(traces_df, prob_init_df, on="id", how="inner")


# nnll_df = pd.read_csv(EXP41_RESULTS_FILE)
# nnll_df['token_ids'] = nnll_df['token_ids'].apply(json.loads)

# df = pd.merge(traces_df, nnll_df, left_on='id', right_on='sample_id', how='inner')

In [3]:
# def get_thinking_trace_from_completion(completion: str) -> str:
#     prefix_split_str = '<|im_start|>assistant\n'
#     completion_with_suffix = completion.split(prefix_split_str)[-1].strip()

#     suffix_split_str = '<|im_end|>'
#     completion = completion_with_suffix.split(suffix_split_str)[0].strip()
#     return completion


# def get_token_length(text: str, tokenizer: PreTrainedTokenizer) -> int:
#     token_ids = tokenizer.encode(text, add_special_tokens=False)
#     return len(token_ids)


# df["partial_trace"] = df["pivotal_context"].apply(get_thinking_trace_from_completion)
# df["partial_trace_token_length"] = df["partial_trace"].apply(
#     lambda x: get_token_length(x, tokenizer)
# )
# df["trace_token_length"] = df["trace"].apply(lambda x: get_token_length(x, tokenizer))
# df["pivotal_token_relative_position"] = (
#     (df["partial_trace_token_length"] + 1) / df["trace_token_length"]
# )

# df = df.drop(columns=['pivotal_context'])
# df = df.drop(columns=['trace', 'metadata', 'partial_trace'])

# df.to_csv(PREPROCESSED_FILE, index=False)

In [4]:
def parse_token_ids(value: object) -> list[int]:
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return []


if PREPROCESSED_FILE.exists():
    prep_df = pd.read_csv(PREPROCESSED_FILE)
    prep_df = prep_df.drop_duplicates()
    prep_df["token_ids"] = prep_df["token_ids"].apply(parse_token_ids)
else:
    prep_df = df.copy()

In [5]:
POSITION_BIN_SIZE = 0.05
# NNLL_PERCENTILES = [10, 25, 75, 90]


def plot_nnll_by_position(df: pd.DataFrame, title_suffix: str | None = None) -> go.Figure:
    df = df.copy()
    bins = np.arange(0, 1 + POSITION_BIN_SIZE, POSITION_BIN_SIZE)
    bin_labels = np.round(bins[1:], 10)

    position_bins = pd.cut(df["pivotal_token_relative_position"], bins=bins, include_lowest=True)

    agg_df = df.groupby(position_bins, observed=True)["nnll"].agg(
        mean="mean",
        median="median",
        std="std",
        # **{f"p{p}": lambda x, p=p: x.quantile(p / 100) for p in NNLL_PERCENTILES},
    )
    agg_df.index = bin_labels
    agg_df.index.name = "position_bin"
    agg_df = agg_df.reset_index()

    traces_config = [
        # *[(f"p{p}", "dot", 1) for p in NNLL_PERCENTILES],
        ("median", "dash", 2),
        ("mean", None, 2),
    ]

    std_color = "rgba(99, 110, 250, 0.15)"
    traces = [
        go.Scatter(
            x=agg_df["position_bin"], y=agg_df[name],
            mode="lines", name=name,
            line=dict(dash=dash, width=width),
        )
        for name, dash, width in traces_config
    ]
    traces.extend([
        go.Scatter(
            x=agg_df["position_bin"], y=agg_df["mean"] - agg_df["std"],
            mode="lines", line=dict(width=0), showlegend=False,
        ),
        go.Scatter(
            x=agg_df["position_bin"], y=agg_df["mean"] + agg_df["std"],
            mode="lines", name="mean ± std",
            line=dict(width=0),
            fill="tonexty", fillcolor=std_color,
        ),
    ])

    fig = go.Figure(traces)
    title = "Agg. NNLL by token relative position"
    if title_suffix:
        title += title_suffix
    fig.update_layout(
        title=title,
        xaxis_title="Relative position (bin)",
        yaxis_title="NNLL",
        xaxis=dict(tickvals=bin_labels),
        height=400,
        width=1200
    )
    return fig


plot_nnll_by_position(prep_df)

In [6]:
fig = plot_nnll_by_position(prep_df[prep_df['prob_init'].between(0.0, 0.3)], title_suffix=", prob. init between [0.0, 0.3]")
fig.show()

fig = plot_nnll_by_position(prep_df[prep_df['prob_init'].between(0.3, 0.6)], title_suffix=", prob. init between [0.3, 0.6]")
fig.show()

fig = plot_nnll_by_position(prep_df[prep_df['prob_init'].between(0.6, 1.0)], title_suffix=", prob. init between [0.6, 1.0]")
fig.show()

In [7]:
def smooth_rolling(s: pd.Series, min_window_samples=5, window_samples_ratio=0.05) -> pd.Series:
    window = min(min_window_samples, int(len(s) * window_samples_ratio))
    return s.rolling(window=window, center=True, min_periods=1).mean()

prep_df["nnll_rolling"] = prep_df.groupby("sample_id")["nnll"].transform(smooth_rolling)

In [8]:
sample_ids = list(prep_df["sample_id"].unique())[:50]
sample_ids = ["5ae54b6355429908b63265cc"]
len(sample_ids)

1

In [9]:
plot_df = prep_df[prep_df["sample_id"].isin(sample_ids)].copy()
plot_df = plot_df.sort_values(['sample_id', 'pivotal_token_relative_position'])

In [10]:
melted_df = plot_df.sort_values(['sample_id', 'pivotal_token_relative_position']).melt(
    id_vars=['sample_id', 'pivotal_token_relative_position', 'span_text', 'ground_truth', 'query', 'prob_init'],
    value_vars=['nnll', 'nnll_rolling'],
    var_name='metric',
    value_name='value',
)

fig = px.line(

    melted_df,
    x='pivotal_token_relative_position',
    y='value',
    color='sample_id',
    line_dash='metric',
    line_dash_map={'nnll': 'dash', 'nnll_rolling': 'solid'},
    custom_data=['span_text', 'ground_truth', 'query', 'prob_init'],
    title="NNLL and Smoothed NNLL by token relative position",
    labels={'pivotal_token_relative_position': 'Relative Position', 'value': 'NNLL'},
    height=500,
    width=1300,
)
fig.update_traces(hovertemplate=(
    "<b>Position:</b> %{x:.3f}<br>"
    "<b>NNLL:</b> %{y:.4f}<br>"
    "<b>Prob. init:</b> %{customdata[3]:.3f}<br>"
    # "<b>Query:</b> %{customdata[2]}<br>"
    "<b>Ground Truth:</b> %{customdata[1]}<br>"
    "<b>Span:</b> %{customdata[0]}<extra></extra>"
))
fig.show()